In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import io, filters, measure, morphology, transform
from scipy import ndimage as ndi
from tqdm.notebook import tqdm

def analyze_phagocytosis_raw_intensity(image_path, sample_name, pixels_per_micron=9.625, min_area=300):
    """
    Quantifies total raw RFP intensity per macrophage as a function of distance.
    """
    try:
        # 1. LOAD DATA
        img = io.imread(image_path)
        red_raw = img[0] if img.shape[0] == 2 else img[:,:,0]
        green_raw = img[1] if img.shape[0] == 2 else img[:,:,1]

        # 2. SPHEROID DETECTION (Find Center)
        ds = 4
        red_ds = transform.rescale(red_raw, 1/ds, anti_aliasing=True)
        s_mask_ds = red_ds > (filters.threshold_otsu(red_ds) * 0.9)
        labels = measure.label(s_mask_ds)
        regions = measure.regionprops(labels)
        if not regions: return None
        largest_s = max(regions, key=lambda x: x.area)
        cy, cx = np.array(largest_s.centroid) * ds

        #RFP Processing
        rfp_bg = filters.median(red_raw, morphology.disk(20))
        rfp_corrected = np.where(red_raw > rfp_bg, red_raw - rfp_bg, 0)
        
        #
        g_thresh = filters.threshold_otsu(green_raw)
        g_mask = morphology.binary_closing(green_raw > g_thresh, morphology.disk(3))
        g_mask = ndi.binary_fill_holes(g_mask)
        labeled_macs = measure.label(g_mask)

       #Intenstiy Quantification
        cell_data = []
        for prop in measure.regionprops(labeled_macs):
            if prop.area > min_area:
                min_r, min_c, max_r, max_c = prop.bbox
                
                # Extract RFP signal specifically within the green mask
                cell_rfp_crop = rfp_corrected[min_r:max_r, min_c:max_c]
                internal_values = cell_rfp_crop[prop.image]
                
                # RAW TOTAL INTENSITY
                total_raw_intensity = np.sum(internal_values)
                
                dist_px = np.sqrt((prop.centroid[0]-cy)**2 + (prop.centroid[1]-cx)**2)
                
                cell_data.append({
                    'File_Name': sample_name,
                    'Distance_um': dist_px / pixels_per_micron,
                    'Phago_Score': total_raw_intensity,
                    'Area_px': prop.area
                })
        return pd.DataFrame(cell_data)
    except Exception as e:
        print(f"Error in {sample_name}: {e}")
        return None

# --- BATCH EXECUTION ---
data_dir = '/Volumes/Roh-Johnson_Lab/Daniel/Data/Confocal Microscopy/2026/3-24-2026-262-24hr-matrigelhighresphago/Processed/analysis/'
file_list = sorted(glob.glob(os.path.join(data_dir, "MAX_*.tif")))

all_results = []
for file_path in file_list:
    name = os.path.basename(file_path).replace(".tif", "")
    print(f"Processing: {name}") # Manual status update
    df = analyze_phagocytosis_raw_intensity(file_path, name)
    if df is not None:
        all_results.append(df)

if all_results:
    master_df = pd.concat(all_results, ignore_index=True)
    
    # Define Group based on keywords
    def get_group(name):
        if 'GFP' in name: return 'GFP'
        if '763hFC-E62K' in name: return '763hFC-E62K'
        return 'Unknown'
    
    master_df['Group'] = master_df['File_Name'].apply(get_group)
    master_df['Log_Phago_Score'] = np.log10(master_df['Phago_Score'] + 1)
    
    #Final export
    master_df.to_csv(os.path.join(data_dir, "batch_analysis_raw_intensity.csv"), index=False)
    print("\nBatch analysis complete. Raw Intensity data saved.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import ttest_ind

# 1. LOAD DATA
csv_path = '/Volumes/Roh-Johnson_Lab/Daniel/Data/Confocal Microscopy/2026/3-24-2026-262-24hr-matrigelhighresphago/Processed/analysis/batch_analysis_raw_intensity.csv'
df = pd.read_csv(csv_path)

# Calculate Log Score
df['Log_Phago_Score'] = np.log10(df['Phago_Score'] + 1)

# --- LOGIC: RENAME & ORDER GROUPS ---
df['Group'] = df['Group'].replace('763hFC-E62K', 'hFC-E62K')
plot_order = ['GFP', 'hFC-E62K']
custom_palette = {'GFP': '#2dc9d2', 'hFC-E62K': '#000000'}

# Illustrator Compatibility Settings
plt.rcParams['svg.fonttype'] = 'none' # Keeps text as editable text in Illustrator
plt.rcParams['pdf.fonttype'] = 42

# 2. PLOT 1: SPATIAL PROFILE (DOTS WITH BORDERS + GRID)
sns.set_theme(style="whitegrid")
g = sns.JointGrid(data=df, x="Distance_um", y="Log_Phago_Score", hue="Group", 
                  hue_order=plot_order, palette=custom_palette,
                  height=4,   # Total size of the square figure in inches
                  ratio=5,    # Ratio of main plot height to marginal plot height
                  space=0.2)  # Gap between the main plot and the marginals

# Plot dots with black borders
g.plot_joint(sns.scatterplot, s=10, alpha=0.8, edgecolor='black', linewidth=0.3)

# Marginal histograms
g.plot_marginals(sns.histplot, kde=True, alpha=0.8, common_norm=False)

# Formatting
g.ax_joint.set_xlabel("Distance from Spheroid Center (um)", fontsize=8, fontweight='bold')
g.ax_joint.set_ylabel("Total Raw RFP Intensity (Log10)", fontsize=8, fontweight='bold')
g.ax_joint.grid(True, linestyle='--', alpha=0.6)

# Export Spatial Profile
spatial_svg = os.path.join(os.path.dirname(csv_path), "Spatial_Profile_Publication.svg")
plt.savefig(spatial_svg, format='svg', bbox_inches='tight')
plt.show()
print(f"Spatial Profile exported to: {spatial_svg}")


# 3. PLOT 2: REPLICATE MFI SUMMARY (BOX & WHISKERS)
image_stats = df.groupby(['File_Name', 'Group'])['Phago_Score'].median().reset_index()
image_stats['Log_MFI'] = np.log10(image_stats['Phago_Score'] + 1)

plt.figure(figsize=(4, 6))
sns.set_theme(style="whitegrid")

# Box plot
sns.boxplot(data=image_stats, x='Group', y='Log_MFI', order=plot_order, 
            palette=custom_palette, width=0.5, showfliers=False, 
            linewidth=1.5, boxprops=dict(edgecolor='black'))

# Replicate dots (N=10)
sns.swarmplot(data=image_stats, x='Group', y='Log_MFI', order=plot_order,
              color='black', size=10, edgecolor='white', linewidth=1)

# Stats
grp_a = image_stats[image_stats['Group'] == 'GFP']['Log_MFI']
grp_b = image_stats[image_stats['Group'] == 'hFC-E62K']['Log_MFI']
t_stat, p_val = ttest_ind(grp_a, grp_b)

plt.title(f"Replicate Median Intensity (N=10 Images)\np = {p_val:.4f}", fontsize=14, fontweight='bold')
plt.ylabel("Median Image Intensity (Log10)", fontsize=12, fontweight='bold')
plt.xlabel("Group", fontsize=12, fontweight='bold')
plt.tight_layout()

# Export Replicate Summary
replicate_svg = os.path.join(os.path.dirname(csv_path), "Replicate_Summary_Publication.svg")
plt.savefig(replicate_svg, format='svg', bbox_inches='tight')
plt.show()
print(f"Replicate Summary exported to: {replicate_svg}")